In [1]:
# Core LLM (ChatGPT / OpenAI)
from langchain_openai import ChatOpenAI

# Tools (to define each agent step)
from langchain.tools import tool

# LangGraph for creating agent workflows
from langgraph.graph import StateGraph, END

# Core schema (used for structured outputs)
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate

# Environment management
from dotenv import load_dotenv

# Utilitiesx
import os, re, pandas as pd

C:\Anaconda3\envs\genai_translation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================================================
# 2️⃣ Define AI Agent Tools (Each function = one Agent)
# =========================================================

from langchain.tools import tool
import re
from langdetect import detect,DetectorFactory
from transformers import pipeline

# 1️⃣ Preprocessing Agent — clean and normalize text
@tool
def preprocess_tool(text: str) -> str:
    """Cleans the input text by removing unwanted characters and spaces."""
    cleaned = re.sub(r'[^\w\s,.!?]', '', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


# 2️⃣ Language Detection Agent — identify input language
@tool
def language_detect_tool(text: str) -> str:
    """
    Detects the language of the given text using langdetect.
    Example: 'Bonjour' → 'fr'
    """
    try:
        lang = detect(text)
    except:
        lang = "unknown"
    return lang


# 3️⃣ Context Analysis Agent — Domain + Tone Detection
# Load once (outside the function)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
domain_labels = ["Business", "Research", "Education", "Technology", "Medical", "Legal", "News"]

@tool
def context_tool(text: str) -> str:
    """
    Detects:
    1️.Tone → Positive / Negative / Neutral
    2.Domain → Business / Education / etc.
    Returns both with confidence score.
    """
    text_lower = text.lower()

    # Tone detection
    if any(w in text_lower for w in ["happy", "great", "good", "awesome", "joy", "love"]):
        tone = "Positive tone"
    elif any(w in text_lower for w in ["sad", "bad", "angry", "upset", "hate", "tired"]):
        tone = "Negative tone"
    else:
        tone = "Neutral tone"

    # Ignore short/casual chats
    casual_words = ["hi", "hello", "ok", "bye", "thanks", "thank you", "hmm", "hey", "yo"]
    if text_lower.strip() in casual_words or len(text.split()) < 3:
        return f"Casual/Conversational text detected. ({tone})"

    #Domain detection
    result = classifier(text, domain_labels)
    domain = result["labels"][0]
    confidence = result["scores"][0]

    if confidence >= 0.6:
        return f"Domain: {domain} (Confidence: {confidence:.2f}) | {tone}"
    else:
        return f"Uncertain domain (Confidence: {confidence:.2f}) | {tone}"
'''
@tool
def context_tool(text: str) -> str:
    """Detects the tone (Positive, Negative, Neutral) of the text."""
    text_lower = text.lower()
    if any(w in text_lower for w in ["happy", "great", "good", "awesome", "joy"]):
        return "Positive tone"
    elif any(w in text_lower for w in ["sad", "bad", "angry", "upset", "hate"]):
        return "Negative tone"
    else:
        return "Neutral tone"
'''

# 4️⃣ Moderation Agent — flag unsafe content
@tool
def moderation_tool(text: str) -> str:
    """Flags text if it contains unsafe or offensive words."""
    banned_words = ["kill", "attack", "hate", "terror", "violence"]
    return "Flagged" if any(word in text.lower() for word in banned_words) else "Safe"

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
C:\Anaconda3\envs\genai_translation\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this articl

In [3]:
# =========================================================
# ✅ Translation Function using ChatGPT (LangChain)
# =========================================================
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()
llm_translate = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

def real_translation_tool(text, target_lang):
    prompt = f"Translate the following English text to {target_lang}: {text}"
    response = llm_translate.invoke([HumanMessage(content=prompt)])
    return response.content

In [4]:
# =========================================================
# ✅ LangGraph Pipeline for AI Agents
# =========================================================

from langgraph.graph import StateGraph

# -------------------------
# Step 1️⃣ - Preprocessing Node
# -------------------------
def node_preprocess(state):
    text = state["text"]
    cleaned = preprocess_tool.func(text)
    return {**state, "cleaned_text": cleaned}


# -------------------------
# Step 2️⃣ - Language Detection Node
# -------------------------
DetectorFactory.seed = 0  # To make results consistent
def node_langdetect(state):
    text = state["cleaned_text"]
    try:
        if len(text.split()) < 4:  # too short → fallback
            detected_lang = "en"
        else:
            detected_lang = detect(text)
    except:
        detected_lang = "en"  # fallback if detection fails

    new_state = {**state, "detected_lang": detected_lang}
    return new_state


# -------------------------
# Step 3️⃣ - Context Analysis Node
# -------------------------
def node_context(state):
    text = state["cleaned_text"]
    tone = context_tool.func(text)
    return {**state, "tone": tone}


# -------------------------
# Step 4️⃣ - Moderation Node
# -------------------------
def node_moderation(state):
    text = state["cleaned_text"]
    moderation_status = moderation_tool.func(text)
    return {**state, "moderation_status": moderation_status}


# -------------------------
# Step 5️⃣ - Translation Node
# -------------------------
def node_translation(state):
    text = state["cleaned_text"]
    target_lang = state.get("target_language", "ta")

    if state["moderation_status"] == "Safe":
        if state["detected_lang"] == target_lang:
            translated = "Already in target language."
        else:
            translated = real_translation_tool(text, target_lang)
    else:
        translated = "Translation skipped due to unsafe content."

    return {**state, "translated_text": translated}


# -------------------------
# Build LangGraph
# -------------------------
graph = StateGraph(dict)
graph.add_node("Preprocess", node_preprocess)
graph.add_node("LanguageDetection", node_langdetect)
graph.add_node("ContextAnalysis", node_context)
graph.add_node("Moderation", node_moderation)
graph.add_node("Translation", node_translation)

graph.add_edge("Preprocess", "LanguageDetection")
graph.add_edge("LanguageDetection", "ContextAnalysis")
graph.add_edge("ContextAnalysis", "Moderation")
graph.add_edge("Moderation", "Translation")

graph.set_entry_point("Preprocess")
graph.set_finish_point("Translation")

app = graph.compile()

# ✅ expose it for import
def run_translation_graph(initial_state):
    """Reusable LangGraph pipeline"""
    return app.invoke(initial_state)

In [8]:
if __name__ == "__main__":
    # Ask user for input text
    user_text = input("Enter your text to translate: ")
    
    # Ask target language (optional)
    target_language = input("Enter target language code (like 'ta' for Tamil, 'fr' for French): ")
    
    # Create initial state for LangGraph
    initial_state = {"text": user_text, "target_language": target_language}
    
    # Run graph
    final_state = app.invoke(initial_state)
    
print("Cleaned Text:", final_state["cleaned_text"])
print("Detected Language:", final_state["detected_lang"])
print("Tone:", final_state["tone"])
print("Moderation:", final_state["moderation_status"])
print("Translation:", final_state["translated_text"])

Enter your text to translate:  Our company achieved record profits this year
Enter target language code (like 'ta' for Tamil, 'fr' for French):  ta


Cleaned Text: Our company achieved record profits this year
Detected Language: en
Tone: Domain: Business (Confidence: 0.78) | Neutral tone
Moderation: Safe
Translation: Our company achieved record profits this year. 

In Tamil, this can be translated as: "எங்கள் நிறுவனம் இந்த ஆண்டில் சாதாரண வருமானங்களை அடைந்தது."


In [ ]:
#Need to Check  Domain